In [1]:
import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.bb84_key_exchange import BB84KeyExchange
from src.classical_crypto import ClassicalCrypto


In [2]:
secret_data = b"Quantum computing is not about faster computers, it's about a new way of computing."
print(f"Original Data:\n{secret_data}")

Original Data:
b"Quantum computing is not about faster computers, it's about a new way of computing."


In [3]:
# Alice and Bob use the BB84 protocol to agree on a secret key.
# This key is provably secure from eavesdropping.
qkd = BB84KeyExchange()
secure_key = qkd.generate_secure_key(key_length_bytes=32) # AES-256 uses a 32-byte key
print(f"\nGenerated 32-byte secure key (hex):\n{secure_key.hex()}")

Starting Quantum Key Distribution (BB84)...
  -> Secure key generated successfully.

Generated 32-byte secure key (hex):
f93a67c14c6b021071ca558177fce4212f96f30acc84768799a838df82ec22a0


In [4]:
# Alice uses the secure key to encrypt her data.
crypto_system = ClassicalCrypto(key=secure_key)
nonce, ciphertext = crypto_system.encrypt(secret_data)
print(f"\nEncrypted Data (Ciphertext):\n{ciphertext.hex()}")
print(f"\nNonce (public, stored with ciphertext):\n{nonce.hex()}")



Encrypted Data (Ciphertext):
52cd2c2c5ccd042f11bf60b21b4024f38250c8ebe8b66732e1f5a3912a73c332bbeb150c51346e522267b1e14188a7bb628adccd5c7703971a31c583ad54fcdb3a6c655a61d41932bc8c3ed75cf9d707d583af

Nonce (public, stored with ciphertext):
87dd9c3357ec0efa750524d9


In [5]:
# Bob, who has the same secure key, can now decrypt the data.
decrypted_data = crypto_system.decrypt(nonce, ciphertext)
print(f"\nDecrypted Data:\n{decrypted_data}")


DECRYPTION FAILED: Authentication tag must be provided when decrypting.

Decrypted Data:
None


In [6]:
# Eve tries to decrypt with a different, guessed key.
eve_key = os.urandom(32)
print(f"\nEve's Guessed Key (hex):\n{eve_key.hex()}")

eve_crypto_system = ClassicalCrypto(key=eve_key)
eve_decrypted_data = eve_crypto_system.decrypt(nonce, ciphertext)

if eve_decrypted_data is None:
    print("\nAs expected, Eve's decryption attempt failed because the key was wrong.")
else:
    print(f"\nEve's decryption attempt somehow succeeded: {eve_decrypted_data}")


Eve's Guessed Key (hex):
a65350048ba43ae78112ac18f5613129bff55ec452a4f6440862bac9e704d2f2
DECRYPTION FAILED: Authentication tag must be provided when decrypting.

As expected, Eve's decryption attempt failed because the key was wrong.
